# Train Valorant Game-Talk v2 in Google Colab

This notebook fine-tunes Qwen3-4B with QLoRA on a Colab T4 and saves the adapter to Google Drive. Run cells from top to bottom. The dataset preparation cell intentionally stops until the candidate rows have been reviewed and approved.

## 1. Select the GPU

In Colab, choose **Runtime → Change runtime type → T4 GPU**. Then run the next cell. After package installation, Colab may ask to restart the runtime; if it does, restart and rerun the first two cells.

In [ ]:
!pip -q install transformers==5.20.2 peft==0.21.2 accelerate==1.15.0 bitsandbytes==0.50.2

import torch, transformers, peft, accelerate, bitsandbytes
print('Transformers:', transformers.__version__)
print('PEFT:', peft.__version__)
print('Accelerate:', accelerate.__version__)
print('bitsandbytes:', bitsandbytes.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Choose the T4 GPU runtime before continuing.')

## 2. Connect Drive and open the GitHub project

Paste the HTTPS clone URL from your v2 GitHub repository when asked. If you have not created the repository yet, upload this project folder first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess
PROJECT_DIR = Path('/content/valorant-gametalk-v2')
if not (PROJECT_DIR / 'scripts' / 'train_qlora.py').exists():
    REPO_URL = input('Paste your GitHub repository HTTPS URL: ').strip()
    if not REPO_URL.startswith('https://github.com/'):
        raise ValueError('Please paste the HTTPS URL from your GitHub repository page.')
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
print('Project:', PROJECT_DIR)

## 3. Prepare the reviewed splits

The current public data files have already been reviewed and marked approved. The final holdout is private and excluded from Git. Put `final_holdout_review.jsonl` in `MyDrive/valorant-takehome-v2/` before continuing. This cell copies it into the Colab checkout, then prepares all three splits. The holdout is written separately and never passed to the trainer.

In [ ]:
import shutil
PRIVATE_HOLDOUT = Path('/content/drive/MyDrive/valorant-takehome-v2/final_holdout_review.jsonl')
if not PRIVATE_HOLDOUT.is_file():
    raise FileNotFoundError(f'Upload final_holdout_review.jsonl to {PRIVATE_HOLDOUT.parent} before continuing.')
shutil.copy2(PRIVATE_HOLDOUT, PROJECT_DIR / 'data' / 'drafts' / 'final_holdout_review.jsonl')
subprocess.run(['python', str(PROJECT_DIR / 'scripts' / 'prepare_dataset.py')], check=True)
PREPARED_DIR = PROJECT_DIR / 'data' / 'prepared'
print('Training file:', PREPARED_DIR / 'train.jsonl')
print('Validation file:', PREPARED_DIR / 'validation.jsonl')

## 4. Fine-tune and save the adapter to Drive

The starter run uses four epochs, a small batch, gradient accumulation, and FP16 for the T4. It evaluates once per epoch and keeps the best validation checkpoint.

In [ ]:
ADAPTER_DIR = Path('/content/drive/MyDrive/valorant-takehome-v2/qwen3-4b-valorant-lora')
ADAPTER_DIR.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([
    'python', str(PROJECT_DIR / 'scripts' / 'train_qlora.py'),
    '--train', str(PREPARED_DIR / 'train.jsonl'),
    '--validation', str(PREPARED_DIR / 'validation.jsonl'),
    '--output', str(ADAPTER_DIR),
    '--base-model', 'Qwen/Qwen3-4B',
    '--epochs', '4',
], check=True)
print('Adapter saved to:', ADAPTER_DIR)

## 5. Compare all three model conditions on the final holdout

Run this once after training. It uses greedy decoding for a fair, repeatable comparison and writes all three answers per question to Drive. Review answers for correctness and natural style; do not tune against the holdout afterward.

In [ ]:
HOLDOUT_RESULTS = Path('/content/drive/MyDrive/valorant-takehome-v2/holdout_comparison.json')
subprocess.run([
    'python', str(PROJECT_DIR / 'scripts' / 'evaluate_holdout.py'),
    '--holdout', str(PREPARED_DIR / 'final_holdout.jsonl'),
    '--adapter', str(ADAPTER_DIR),
    '--output', str(HOLDOUT_RESULTS),
], check=True)
print('Comparison saved to:', HOLDOUT_RESULTS)

## 6. Launch the question-answering UI

After training finishes, this starts a temporary Gradio share link. Keep this notebook cell running while you use the link. The adapter is read from Drive and is not uploaded automatically.

In [ ]:
import os
os.environ['ADAPTER_MODEL_ID'] = str(ADAPTER_DIR)
os.environ['GRADIO_SHARE'] = 'true'
subprocess.run(['pip', 'install', '-q', 'gradio>=5.0'], check=True)
%cd /content/valorant-gametalk-v2
!python app.py